### Deep learning model training.
#### Top accuracy(inner 512x512):
|model|miou|oa|date|     
|----|----|----|----|     
|deeplabv3+|0.912|0.948|20260428|    
|deeplabv3+(mb_v2)|0.923|0.962|20260428|  
||||    

#### Top accuracy(inner 448x448):
|model|miou|oa|date|     
|----|----|----|----|     
|deeplabv3+|0.888|0.924|20260428|    
|deeplabv3+(mb_v2)|0.904|0.943|20260428|  
||||    

### conclusions:   
1. 




In [2]:
import time
import torch
import random
import pandas as pd
import torch.nn as nn
from glob import glob
from notebooks import config
import torch.nn.functional as F
import matplotlib.pyplot as plt
from utils.imgShow import imsShow
from torchvision.transforms import v2
from utils.data_aug import GaussianNoise
from utils.dataloader import read_scenes 
from utils.metrics import oa_binary, miou_binary
from utils.dataloader import SceneArraySet, PatchPathSet
# import segmentation_models_pytorch as smp
from model import u2net_timm
from model import deeplabv3plus, deeplabv3plus_mobilev2
from github_repo.SSRS.ASMFNet.models.swinfusenet.vision_transformer import SwinFuseNet
from model import CMFNet


In [3]:
patch_size = 512     ## patch size setting
patch_resize = None  ## patch resize setting
### traset
paths_scene_tra, paths_truth_tra = config.paths_scene_tra, config.paths_truth_tra
paths_dem_tra = config.paths_dem_tra
print(f'train scenes: {len(paths_scene_tra)}')
## valset
paths_valset = sorted(glob(f'data/dset/valset/patch_{patch_size}/*'))  ## for model prediction 
print(f'vali patch {patch_size}: {len(paths_valset)}')


train scenes: 52
vali patch 512: 117


### dataset loading

In [4]:
scenes_arr, truths_arr = read_scenes(paths_scene_tra, 
                                     paths_truth_tra, 
                                     paths_dem_tra) 


In [5]:
transforms_tra = v2.Compose([
            v2.ToImage(),
            v2.RandomCrop(size=(patch_size, patch_size)),
            v2.RandomHorizontalFlip(p=0.3),
            v2.RandomVerticalFlip(p=0.3),
            v2.RandomApply([v2.RandomRotation(degrees=15)], p=0.3),
            GaussianNoise(mean = 0.0, sigma_max_img=0.1, sigma_max_dem=0, p=0.3) 
            ])

transforms_val = v2.Compose([
      v2.ToDtype(torch.float32),
       ])  


In [6]:
# Create dataset instances
tra_data = SceneArraySet(scenes_arr=scenes_arr, 
                          truths_arr=truths_arr, 
                          patch_size=patch_size,
                          transforms=transforms_tra)
val_data = PatchPathSet(paths_valset=paths_valset, transforms=transforms_val)

## Create data loaders
tra_loader = torch.utils.data.DataLoader(tra_data, 
                                         batch_size=4, 
                                         shuffle=True, 
                                         num_workers=5)
val_loader = torch.utils.data.DataLoader(val_data, 
                                         batch_size=16, 
                                         shuffle=False,
                                         num_workers=5)


#### Model training

In [7]:
# model = deeplabv3plus(num_bands=7)
# model = deeplabv3plus_mobilev2(num_bands=7)
model = u2net_timm(num_bands_b1=6, num_bands_b2=1, backbone_name='efficientnet_b0')
# model = smp.UnetPlusPlus("resnet34",  
#                   encoder_weights=None,  
#                   in_channels=7,  
#                   classes=1,  
#                   decoder_use_norm=True)
# model = smp.MAnet("resnet34",  
#                   encoder_weights=None,  
#                   in_channels=7,  
#                   classes=1,  
#                   decoder_use_norm=True)
# model = CMFNet(num_bands_b1=6, num_bands_b2=1, patch_size=512)
# model = SwinFuseNet(img_size=512, num_classes=1, num_bands_b1=6, num_bands_b2=1)


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


In [10]:
input_tensor = torch.randn(2, 7, 512, 512)  
output = model(input_tensor)  
output.shape  


torch.Size([2, 1, 512, 512])

In [11]:
### create loss and optimizer
# criterion = nn.BCELoss()
criterion = nn.BCEWithLogitsLoss()
# criterion = GlacierLoss(aux_weight=1)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)  
# lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 
#                                           mode='min', 
#                                           factor=0.5, 
#                                           patience=30)


In [14]:
'''------train step------'''
def train_step(model, loss_fn, optimizer, x, y):
    optimizer.zero_grad()
    # pred, aux1, aux2, aux3  = model(x)
    # loss = loss_fn(pred, y, aux1, aux2, aux3)
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    optimizer.step()
    pred = F.sigmoid(pred)  ## convert logit to prob for metric calculation
    pred = (pred > 0.5).float()  ## convert prob to binary for metric calculation
    miou = miou_binary(pred=pred, truth=y, device=x.device)
    oa = oa_binary(pred=pred, truth=y, device=x.device)
    return loss, miou, oa

'''------validation step------'''
def val_step(model, loss_fn, x, y):
    with torch.no_grad():
        pred = model(x)
        # if x.shape[2] > 256:  ### crop inner 256x256 for evaluation
        #     pred = v2.functional.center_crop(pred, 256)
        #     y = v2.functional.center_crop(y, 256)
    loss = loss_fn(pred, y)
    pred = F.sigmoid(pred)  ## convert logit to prob for metric calculation
    pred = (pred > 0.5).float()  ## convert prob to binary
    miou = miou_binary(pred=pred, truth=y, device=x.device)
    oa = oa_binary(pred=pred, truth=y, device=x.device)
    return loss, miou, oa

'''------train loops------'''
def train_loops(model, loss_fn, 
                    optimizer, 
                    tra_loader, 
                    val_loader,                     
                    epoches, 
                    device, 
                    lr_scheduler=None):
    tra_loss_loops, tra_miou_loops, tra_oa_loops = [], [], []
    val_loss_loops, val_miou_loops, val_oa_loops = [], [], []
    model = model.to(device)
    size_tra_loader = len(tra_loader)
    size_val_loader = len(val_loader)
    for epoch in range(epoches):
        start = time.time()
        tra_loss, val_loss = 0, 0
        tra_miou, val_miou = 0, 0
        tra_oa, val_oa = 0, 0
        '''-----train the model-----'''
        model.train()   
        for x_batch, y_batch in tra_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            loss, miou, oa = train_step(model=model, loss_fn=loss_fn, 
                                            optimizer=optimizer, 
                                            x=x_batch, 
                                            y=y_batch, 
                                            )
            tra_loss += loss.item()
            tra_miou += miou.item()
            tra_oa += oa.item()
        '''----- validation the model: time consuming -----'''
        model.eval()
        if (epoch+1) > 500 and (epoch+1) % 3 == 0: 
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                loss, miou, oa = val_step(model=model, 
                                            loss_fn=loss_fn, 
                                            x=x_batch, 
                                            y=y_batch)
                val_loss += loss.item()
                val_miou += miou.item()
                val_oa += oa.item()
            tra_loss = tra_loss/size_tra_loader
            tra_miou = tra_miou/size_tra_loader
            tra_oa = tra_oa/size_tra_loader
            val_loss = val_loss/size_val_loader
            val_miou = val_miou/size_val_loader
            val_oa = val_oa/size_val_loader
            tra_loss_loops.append(tra_loss); tra_miou_loops.append(tra_miou); tra_oa_loops.append(tra_oa)
            val_loss_loops.append(val_loss); val_miou_loops.append(val_miou); val_oa_loops.append(val_oa)
            print(f'Ep{epoch}: tra-> Loss:{tra_loss:.3f},Oa:{tra_oa:.3f},Miou:{tra_miou:.3f}, '
                    f'val-> Loss:{val_loss:.3f},Oa:{val_oa:.3f}, Miou:{val_miou:.3f},time:{time.time()-start:.1f}s')
        else: 
            tra_loss = tra_loss/size_tra_loader
            tra_miou = tra_miou/size_tra_loader
            tra_oa = tra_oa/size_tra_loader
            print(f'Ep{epoch}: tra-> Loss:{tra_loss:.3f},Oa:{tra_oa:.3f},Miou:{tra_miou:.3f}, \
                                time:{time.time()-start:.1f}s')
        if lr_scheduler:
          lr_scheduler.step(val_loss)    ## if using ReduceLROnPlateau
        ## show the result
        if (epoch+1)%100 == 0:            
            sam_index = random.randrange(len(val_data))
            patch, truth = val_data[sam_index]
            patch, truth = patch.unsqueeze(0).to(device), truth.to(device)
            with torch.no_grad():
                pred = model(patch)
                pred = F.sigmoid(pred)  ## convert logit to prob for visualization
            if patch.shape[2] > 256:  ## zoom in for visualization if patch size > 256
                pred_val = v2.functional.center_crop(pred, 256)
                patch_val = v2.functional.center_crop(patch, 256)
                truth_val = v2.functional.center_crop(truth, 256)
            else:
                patch_val = patch
                pred_val = pred
                truth_val = truth
            ## convert to numpy and plot
            patch = patch[0].to('cpu').detach().numpy().transpose(1,2,0)            
            pred = pred[0].to('cpu').detach().numpy()
            patch_val = patch_val[0].to('cpu').detach().numpy().transpose(1,2,0)
            pred_val = pred_val[0].to('cpu').detach().numpy()
            truth_val = truth_val.to('cpu').detach().numpy()
            imsShow([patch, pred, patch_val, pred_val, truth_val], 
                    clip_list = (2,0,2,0,0),
                    img_name_list=['input_patch', 'pred', 'patch_zoom_in', 'pred_zoom_in', 'truth_zoom_in'], 
                    figsize=(15, 3))
            plt.tight_layout() 
    metrics = {'tra_loss':tra_loss_loops, 'tra_oa': tra_oa_loops, 'tra_miou': tra_miou_loops,
                'val_loss': val_loss_loops, 'val_oa': val_oa_loops, 'val_miou': val_miou_loops}
    return metrics 


In [ ]:
device = torch.device('cuda:0') 
metrics = train_loops(model=model, 
                epoches=1000,  
                loss_fn=criterion,  
                optimizer=optimizer,  
                tra_loader=tra_loader,  
                val_loader=val_loader,  
                # lr_scheduler=lr_scheduler,  
                device=device)  


In [ ]:
# # model saving 
# # model_name = 'unet' 
# # model_name = 'unet_att' 
# model_name = 'u2net_cbam' 
# # model_name = 'deeplabv3plus'  
# # model_name = 'deeplabv3plus_mb2' 
# date_str = time.strftime("%Y-%m-%d-%H", time.localtime())
# date_str = date_str.replace('-', '')  ## remove '-' for file name
# # path_save = f'model/trained/{model_name}/{model_name}_weights_1.pth'
# path_save = f'model/trained/seg_models/{model_name}_weights_{date_str}.pth'
# torch.save(model.state_dict(), path_save)     ## save weights of the trained model 
# ## model.load_state_dict(torch.load(path_save, weights_only=True))  ## load the weights of the trained model
# ## metrics saving
# path_metrics = f'model/trained/seg_models/{model_name}_metrics_{date_str}.csv'    
# ## path_metrics = f'model/trained/{model_name}/{model_name}_metrics_1.csv'    
# metrics_df = pd.DataFrame(metrics)
# metrics_df.to_csv(path_metrics, index=False, sep=',')
